# Классификация Markdown-документов по Excel-справочнику

Пайплайн: **`.md` из папки `output` → Excel-справочник → LLM с reasoning → `№ п/п + тематика` → Excel с результатами**.

PDF на этом этапе уже не используется.


## 1. Импорты

In [ ]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import display


## 2. Настройки и подключение

In [ ]:
BASE_URL = "https://ai-gateway.raisa.go.rshbank.ru/v1"
MODEL = "Qwen/Qwen3.6-35B-test/sovetnik"
API_KEY = "ВАШ_API_KEY"

OUTPUT_FOLDER = Path("./output")
RULES_FILE = Path("./rules.xlsx")
RESULT_FILE = Path("./classification_results.xlsx")

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    timeout=300.0,
)

client.models.list()


## 3. Чтение Excel

Положите Excel со справочником рядом с ноутбуком под именем `rules.xlsx`.


In [ ]:
rules_df = pd.read_excel(RULES_FILE)

print("Столбцы Excel:")
for col in rules_df.columns:
    print("-", repr(col))

display(rules_df.head())


## 4. Укажите точные названия трёх столбцов

In [ ]:
COL_N_PP = "№ п/п"
COL_TOPIC = "Организация и тема документов"
COL_ALGORITHM = "Определение алгоритма в СЭД"

# Если в вашем Excel названия отличаются — замените строки выше.


## 5. Подготовка справочника

In [ ]:
rules_clean = rules_df[
    [COL_N_PP, COL_TOPIC, COL_ALGORITHM]
].copy()

rules_clean = rules_clean.dropna(
    how="all",
    subset=[COL_N_PP, COL_TOPIC, COL_ALGORITHM]
)

for col in [COL_N_PP, COL_TOPIC, COL_ALGORITHM]:
    rules_clean[col] = (
        rules_clean[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

rules_clean = rules_clean[
    (rules_clean[COL_N_PP] != "")
    & (rules_clean[COL_TOPIC] != "")
    & (rules_clean[COL_ALGORITHM] != "")
].reset_index(drop=True)

print("Правил:", len(rules_clean))
display(rules_clean.head())


## 6. Преобразование правил в текст для модели

In [ ]:
def build_rules_text(df):
    parts = []

    for _, row in df.iterrows():
        parts.append(
            f"""№ п/п: {row[COL_N_PP]}
Тематика: {row[COL_TOPIC]}
Алгоритм определения:
{row[COL_ALGORITHM]}"""
        )

    return "\n\n---\n\n".join(parts)


rules_text = build_rules_text(rules_clean)

print(rules_text[:5000])


## 7. Промт классификатора

In [ ]:
CLASSIFICATION_PROMPT = """
Ты классифицируешь документ по справочнику правил.

Тебе переданы:
1. Справочник тематик.
2. Для каждой тематики — № п/п и алгоритм определения.
3. Полный текст одного документа.

Нужно:
- внимательно проанализировать весь документ;
- сопоставить его со всеми правилами;
- учитывать совокупность условий, а не только отдельные слова;
- если правило содержит несколько условий, проверить каждое;
- ничего не придумывать;
- выбрать ровно одну наиболее подходящую тематику;
- вернуть № п/п строго из справочника;
- вернуть тематику строго в формулировке из справочника.

Если ни одно правило не подходит достаточно уверенно, верни:
{
  "n_pp": "Не определено",
  "topic": "Не определено"
}

Ответ — только JSON:
{
  "n_pp": "...",
  "topic": "..."
}
"""


## 8. Функция классификации

In [ ]:
def classify_text(document_text: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": CLASSIFICATION_PROMPT,
            },
            {
                "role": "user",
                "content": f"""СПРАВОЧНИК ТЕМАТИК:

{rules_text}

ТЕКСТ ДОКУМЕНТА:

{document_text}
""",
            },
        ],
        temperature=0,
        response_format={"type": "json_object"},
        extra_body={
            "top_k": 20,
            "chat_template_kwargs": {
                "enable_thinking": True
            },
        },
    )

    raw = response.choices[0].message.content.strip()
    result = json.loads(raw)

    return {
        "n_pp": str(result.get("n_pp", "")).strip(),
        "topic": str(result.get("topic", "")).strip(),
    }


## 9. Проверка результата по Excel

In [ ]:
def validate_result(result: dict) -> dict:
    if (
        result["n_pp"] == "Не определено"
        or result["topic"] == "Не определено"
    ):
        return {
            "n_pp": "Не определено",
            "topic": "Не определено",
        }

    matches = rules_clean[
        (rules_clean[COL_N_PP] == result["n_pp"])
        & (rules_clean[COL_TOPIC] == result["topic"])
    ]

    if matches.empty:
        raise ValueError(
            "Модель вернула пару № п/п / тематика, "
            "которой нет в Excel:\n"
            f"{result}"
        )

    return result


## 10. Markdown-файлы из папки `output`

In [ ]:
md_files = sorted(OUTPUT_FOLDER.glob("*.md"))

print(f"Найдено .md файлов: {len(md_files)}")

for path in md_files[:20]:
    print("-", path.name)


## 11. Тест на одном `.md`

In [ ]:
if not md_files:
    raise FileNotFoundError(
        f"В папке {OUTPUT_FOLDER} нет .md файлов"
    )

test_file = md_files[0]

document_text = test_file.read_text(
    encoding="utf-8"
)

print("Файл:", test_file.name)
print("Символов:", len(document_text))
print()
print(document_text[:3000])


## 12. Классификация тестового файла

In [ ]:
test_result = classify_text(document_text)
test_result = validate_result(test_result)

print("№ п/п:", test_result["n_pp"])
print("Тематика:", test_result["topic"])


## 13. Классификация всех `.md`

In [ ]:
results = []

for i, md_path in enumerate(md_files, start=1):
    print(f"[{i}/{len(md_files)}] {md_path.name}")

    try:
        document_text = md_path.read_text(
            encoding="utf-8"
        )

        result = classify_text(document_text)
        result = validate_result(result)

        results.append({
            "Файл": md_path.name,
            "№ п/п": result["n_pp"],
            "Тематика": result["topic"],
            "Статус": "OK",
        })

    except Exception as e:
        print("Ошибка:", e)

        results.append({
            "Файл": md_path.name,
            "№ п/п": "",
            "Тематика": "",
            "Статус": f"Ошибка: {e}",
        })

results_df = pd.DataFrame(results)

display(results_df)


## 14. Сохранение итогов

In [ ]:
results_df.to_excel(
    RESULT_FILE,
    index=False
)

print(f"Сохранено: {RESULT_FILE}")


## Результат

На выходе создаётся `classification_results.xlsx` с колонками:

- `Файл`
- `№ п/п`
- `Тематика`
- `Статус`

Reasoning включён только для классификации.
